In [1]:
import pandas as pd

df = pd.read_csv('/content/Dataset of Diabetes .csv')
display(df.head())

,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
0,502,17975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,735,34221,M,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,420,47975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,680,87656,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,504,34223,M,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N


In [3]:
print(f"Total missing values in the DataFrame: {df.isnull().sum().sum()}")

Total missing values in the DataFrame: 0


In [4]:
print(df.select_dtypes(include='object').columns)

df = pd.get_dummies(df, columns=['Gender', 'CLASS'], drop_first=True)

print("DataFrame after one-hot encoding:")
display(df.head())

Index(['Gender', 'CLASS'], dtype='object')
DataFrame after one-hot encoding:


,ID,No_Pation,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,Gender_M,Gender_f,CLASS_N,CLASS_P,CLASS_Y,CLASS_Y
0,502,17975,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,False,False,False,False,False,False
1,735,34221,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,True,False,False,False,False,False
2,420,47975,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,False,False,False,False,False,False
3,680,87656,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,False,False,False,False,False,False
4,504,34223,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,True,False,False,False,False,False


In [5]:
print(df.columns)

# Drop one of the Gender columns (e.g., 'Gender_f') to avoid multicollinearity, as 'Gender_M' already represents the binary feature.
df = df.drop(columns=['Gender_f'])

# Drop the duplicate 'CLASS_Y ' column (with trailing space) if it exists.
if 'CLASS_Y ' in df.columns:
    df = df.drop(columns=['CLASS_Y '])

print("DataFrame after correcting one-hot encoding:")
display(df.head())
print("Remaining categorical columns and their types:")
print(df.select_dtypes(include='object').columns)

Index(['ID', 'No_Pation', 'AGE', 'Urea', 'Cr', 'HbA1c', 'Chol', 'TG', 'HDL',
       'LDL', 'VLDL', 'BMI', 'Gender_M', 'Gender_f', 'CLASS_N ', 'CLASS_P',
       'CLASS_Y', 'CLASS_Y '],
      dtype='object')
DataFrame after correcting one-hot encoding:


,ID,No_Pation,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,Gender_M,CLASS_N,CLASS_P,CLASS_Y
0,502,17975,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,False,False,False,False
1,735,34221,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,True,False,False,False
2,420,47975,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,False,False,False,False
3,680,87656,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,False,False,False,False
4,504,34223,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,True,False,False,False


Remaining categorical columns and their types:
Index([], dtype='object')


In [6]:
numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
# Exclude 'ID' and 'No_Pation' as they are identifiers and not subject to outlier treatment in this context.
if 'ID' in numerical_cols:
    numerical_cols.remove('ID')
if 'No_Pation' in numerical_cols:
    numerical_cols.remove('No_Pation')

print(f"Numerical columns for outlier detection: {numerical_cols}")

Numerical columns for outlier detection: ['AGE', 'Urea', 'Cr', 'HbA1c', 'Chol', 'TG', 'HDL', 'LDL', 'VLDL', 'BMI']


In [7]:
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Apply capping
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

    print(f"Outliers in column '{col}' handled using IQR capping. Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}")

print("\nDescriptive statistics of the DataFrame after outlier capping:")
display(df.describe())

Outliers in column 'AGE' handled using IQR capping. Lower Bound: 39.00, Upper Bound: 71.00
Outliers in column 'Urea' handled using IQR capping. Lower Bound: 0.70, Upper Bound: 8.70
Outliers in column 'Cr' handled using IQR capping. Lower Bound: 10.50, Upper Bound: 110.50
Outliers in column 'HbA1c' handled using IQR capping. Lower Bound: 0.95, Upper Bound: 15.75
Outliers in column 'Chol' handled using IQR capping. Lower Bound: 1.60, Upper Bound: 8.00
Outliers in column 'TG' handled using IQR capping. Lower Bound: -0.60, Upper Bound: 5.00
Outliers in column 'HDL' handled using IQR capping. Lower Bound: 0.30, Upper Bound: 1.90
Outliers in column 'LDL' handled using IQR capping. Lower Bound: -0.45, Upper Bound: 5.55
Outliers in column 'VLDL' handled using IQR capping. Lower Bound: -0.50, Upper Bound: 2.70
Outliers in column 'BMI' handled using IQR capping. Lower Bound: 15.50, Upper Bound: 43.50

Descriptive statistics of the DataFrame after outlier capping:


,ID,No_Pation,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI
count,1000.000000,1.000000e+03,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000
mean,340.500000,2.705514e+05,53.986000,4.826843,62.345000,8.280960,4.843420,2.280610,1.142250,2.591640,1.14040,29.566770
std,240.397673,3.380758e+06,7.363968,1.714231,20.297906,2.532224,1.210029,1.150887,0.348675,1.039511,0.62744,4.926358
min,1.000000,1.230000e+02,39.000000,0.700000,10.500000,0.950000,1.600000,0.300000,0.300000,0.300000,0.10000,19.000000
25%,125.750000,2.406375e+04,51.000000,3.700000,48.000000,6.500000,4.000000,1.500000,0.900000,1.800000,0.70000,26.000000
50%,300.500000,3.439550e+04,55.000000,4.600000,60.000000,8.000000,4.800000,2.000000,1.100000,2.500000,0.90000,30.000000
75%,550.250000,4.538425e+04,59.000000,5.700000,73.000000,10.200000,5.600000,2.900000,1.300000,3.300000,1.50000,33.000000
max,800.000000,7.543566e+07,71.000000,8.700000,110.500000,15.750000,8.000000,5.000000,1.900000,5.550000,2.70000,43.500000


In [8]:
from sklearn.preprocessing import MinMaxScaler

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Identify numerical columns for scaling (excluding 'ID' and 'No_Pation')
numerical_cols_to_scale = df.select_dtypes(include=['number']).columns.tolist()
if 'ID' in numerical_cols_to_scale:
    numerical_cols_to_scale.remove('ID')
if 'No_Pation' in numerical_cols_to_scale:
    numerical_cols_to_scale.remove('No_Pation')

# Apply Min-Max scaling to the identified numerical columns
df[numerical_cols_to_scale] = scaler.fit_transform(df[numerical_cols_to_scale])

print("DataFrame after Min-Max scaling:")
display(df.head())

print("Descriptive statistics of the DataFrame after Min-Max scaling:")
display(df.describe())

DataFrame after Min-Max scaling:


,ID,No_Pation,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,Gender_M,CLASS_N,CLASS_P,CLASS_Y
0,502,17975,0.34375,0.500,0.355,0.266892,0.406250,0.127660,1.0000,0.209524,0.153846,0.204082,False,False,False,False
1,735,34221,0.00000,0.475,0.515,0.266892,0.328125,0.234043,0.5000,0.342857,0.192308,0.163265,True,False,False,False
2,420,47975,0.34375,0.500,0.355,0.266892,0.406250,0.127660,1.0000,0.209524,0.153846,0.204082,False,False,False,False
3,680,87656,0.34375,0.500,0.355,0.266892,0.406250,0.127660,1.0000,0.209524,0.153846,0.204082,False,False,False,False
4,504,34223,0.00000,0.800,0.355,0.266892,0.515625,0.148936,0.3125,0.323810,0.115385,0.081633,True,False,False,False


Descriptive statistics of the DataFrame after Min-Max scaling:


,ID,No_Pation,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI
count,1000.000000,1.000000e+03,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,340.500000,2.705514e+05,0.468313,0.515855,0.518450,0.495335,0.506784,0.421406,0.526406,0.436503,0.400154,0.431297
std,240.397673,3.380758e+06,0.230124,0.214279,0.202979,0.171096,0.189067,0.244870,0.217922,0.198002,0.241323,0.201076
min,1.000000,1.230000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,125.750000,2.406375e+04,0.375000,0.375000,0.375000,0.375000,0.375000,0.255319,0.375000,0.285714,0.230769,0.285714
50%,300.500000,3.439550e+04,0.500000,0.487500,0.495000,0.476351,0.500000,0.361702,0.500000,0.419048,0.307692,0.448980
75%,550.250000,4.538425e+04,0.625000,0.625000,0.625000,0.625000,0.625000,0.553191,0.625000,0.571429,0.538462,0.571429
max,800.000000,7.543566e+07,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [12]:
df1 = pd.read_csv('/content/adult.csv')
display(df1.head())

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [13]:
print("First 5 rows of df1:")
display(df1.head())

print("\nConcise summary of df1:")
df1.info()

print("\nMissing values in each column of df1:")
print(df1.isnull().sum())

First 5 rows of df1:


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K



Concise summary of df1:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48842 non-null  int64 
 1   workclass        48842 non-null  object
 2   fnlwgt           48842 non-null  int64 
 3   education        48842 non-null  object
 4   educational-num  48842 non-null  int64 
 5   marital-status   48842 non-null  object
 6   occupation       48842 non-null  object
 7   relationship     48842 non-null  object
 8   race             48842 non-null  object
 9   gender           48842 non-null  object
 10  capital-gain     48842 non-null  int64 
 11  capital-loss     48842 non-null  int64 
 12  hours-per-week   48842 non-null  int64 
 13  native-country   48842 non-null  object
 14  income           48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB

Missing values in each column of df1:
age         

In [14]:
import numpy as np

# Replace '?' with np.nan across the entire DataFrame
df1 = df1.replace('?', np.nan)

print("DataFrame after replacing '?' with NaN:")
display(df1.head())

print("\nMissing values in each column of df1 after replacement:")
print(df1.isnull().sum())

DataFrame after replacing '?' with NaN:


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K



Missing values in each column of df1 after replacement:
age                   0
workclass          2799
fnlwgt                0
education             0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64


In [15]:
for col in ['workclass', 'occupation', 'native-country']:
    if df1[col].isnull().sum() > 0:
        mode_value = df1[col].mode()[0]
        df1[col].fillna(mode_value, inplace=True)
        print(f"Missing values in column '{col}' imputed with mode: {mode_value}")

print("\nDataFrame after imputing missing values with mode:")
display(df1.head())

print("\nMissing values in each column of df1 after imputation:")
print(df1.isnull().sum())

Missing values in column 'workclass' imputed with mode: Private
Missing values in column 'occupation' imputed with mode: Prof-specialty
Missing values in column 'native-country' imputed with mode: United-States

DataFrame after imputing missing values with mode:


/tmp/ipython-input-849696792.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df1[col].fillna(mode_value, inplace=True)


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,Private,103497,Some-college,10,Never-married,Prof-specialty,Own-child,White,Female,0,0,30,United-States,<=50K



Missing values in each column of df1 after imputation:
age                0
workclass          0
fnlwgt             0
education          0
educational-num    0
marital-status     0
occupation         0
relationship       0
race               0
gender             0
capital-gain       0
capital-loss       0
hours-per-week     0
native-country     0
income             0
dtype: int64


In [16]:
for col in ['workclass', 'occupation', 'native-country']:
    if df1[col].isnull().sum() > 0:
        mode_value = df1[col].mode()[0]
        df1[col] = df1[col].fillna(mode_value) # Removed inplace=True and assigned back
        print(f"Missing values in column '{col}' imputed with mode: {mode_value}")

print("\nDataFrame after imputing missing values with mode:")
display(df1.head())

print("\nMissing values in each column of df1 after imputation:")
print(df1.isnull().sum())


DataFrame after imputing missing values with mode:


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,Private,103497,Some-college,10,Never-married,Prof-specialty,Own-child,White,Female,0,0,30,United-States,<=50K



Missing values in each column of df1 after imputation:
age                0
workclass          0
fnlwgt             0
education          0
educational-num    0
marital-status     0
occupation         0
relationship       0
race               0
gender             0
capital-gain       0
capital-loss       0
hours-per-week     0
native-country     0
income             0
dtype: int64


In [17]:
print("Original categorical columns in df1:")
print(df1.select_dtypes(include='object').columns)

# Identify categorical columns
categorical_cols_df1 = df1.select_dtypes(include=['object', 'category']).columns.tolist()

# Apply one-hot encoding
df1 = pd.get_dummies(df1, columns=categorical_cols_df1, drop_first=True)

print("\nDataFrame after one-hot encoding:")
display(df1.head())

print("\nShape of DataFrame after one-hot encoding:")
print(df1.shape)

Original categorical columns in df1:
Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'gender', 'native-country', 'income'],
      dtype='object')

DataFrame after one-hot encoding:


,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,...,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia,income_>50K
0,25,226802,7,0,0,40,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
1,38,89814,9,0,0,50,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
2,28,336951,12,0,0,40,True,False,False,False,...,False,False,False,False,False,False,True,False,False,True
3,44,160323,10,7688,0,40,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
4,18,103497,10,0,0,30,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False



Shape of DataFrame after one-hot encoding:
(48842, 98)


In [18]:
numerical_cols_df1 = df1.select_dtypes(include=['number']).columns.tolist()

print(f"Numerical columns for outlier detection in df1: {numerical_cols_df1}")

for col in numerical_cols_df1:
    Q1 = df1[col].quantile(0.25)
    Q3 = df1[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Apply capping
    df1[col] = df1[col].clip(lower=lower_bound, upper=upper_bound)

    print(f"Outliers in column '{col}' handled using IQR capping. Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}")

print("\nDescriptive statistics of df1 after outlier capping:")
display(df1.describe())

Numerical columns for outlier detection in df1: ['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Outliers in column 'age' handled using IQR capping. Lower Bound: -2.00, Upper Bound: 78.00
Outliers in column 'fnlwgt' handled using IQR capping. Lower Bound: -62586.75, Upper Bound: 417779.25
Outliers in column 'educational-num' handled using IQR capping. Lower Bound: 4.50, Upper Bound: 16.50
Outliers in column 'capital-gain' handled using IQR capping. Lower Bound: 0.00, Upper Bound: 0.00
Outliers in column 'capital-loss' handled using IQR capping. Lower Bound: 0.00, Upper Bound: 0.00
Outliers in column 'hours-per-week' handled using IQR capping. Lower Bound: 32.50, Upper Bound: 52.50

Descriptive statistics of df1 after outlier capping:


,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,48842.000000,48842.000000,48842.0,48842.0,48842.000000
mean,38.618566,186770.707163,10.122088,0.0,0.0,41.192805
std,13.630359,95328.614282,2.456895,0.0,0.0,6.184799
min,17.000000,12285.000000,4.500000,0.0,0.0,32.500000
25%,28.000000,117550.500000,9.000000,0.0,0.0,40.000000
50%,37.000000,178144.500000,10.000000,0.0,0.0,40.000000
75%,48.000000,237642.000000,12.000000,0.0,0.0,45.000000
max,78.000000,417779.250000,16.000000,0.0,0.0,52.500000


In [19]:
from sklearn.preprocessing import MinMaxScaler

# Identify numerical columns for scaling
numerical_cols_df1_scaled = df1.select_dtypes(include=['number']).columns.tolist()

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Apply Min-Max scaling to the identified numerical columns
df1[numerical_cols_df1_scaled] = scaler.fit_transform(df1[numerical_cols_df1_scaled])

print("DataFrame after Min-Max scaling:")
display(df1.head())

print("Descriptive statistics of the DataFrame after Min-Max scaling:")
display(df1.describe())

DataFrame after Min-Max scaling:


,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,...,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia,income_>50K
0,0.131148,0.529026,0.217391,0.0,0.0,0.375,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
1,0.344262,0.191196,0.391304,0.0,0.0,0.875,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
2,0.180328,0.800667,0.652174,0.0,0.0,0.375,True,False,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0.442623,0.365080,0.478261,0.0,0.0,0.375,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
4,0.016393,0.224940,0.478261,0.0,0.0,0.000,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False


Descriptive statistics of the DataFrame after Min-Max scaling:


,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,48842.000000,48842.000000,48842.0,48842.0,48842.00000
mean,0.354403,0.430304,0.488877,0.0,0.0,0.43464
std,0.223449,0.235092,0.213643,0.0,0.0,0.30924
min,0.000000,0.000000,0.000000,0.0,0.0,0.00000
25%,0.180328,0.259598,0.391304,0.0,0.0,0.37500
50%,0.327869,0.409030,0.478261,0.0,0.0,0.37500
75%,0.508197,0.555759,0.652174,0.0,0.0,0.62500
max,1.000000,1.000000,1.000000,0.0,0.0,1.00000


In [20]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns for scaling
numerical_cols_df1_standard_scaled = df1.select_dtypes(include=['number']).columns.tolist()

# Initialize the StandardScaler
scaler = StandardScaler()

# Apply Standard scaling to the identified numerical columns
df1[numerical_cols_df1_standard_scaled] = scaler.fit_transform(df1[numerical_cols_df1_standard_scaled])

print("DataFrame after Standard scaling:")
display(df1.head())

print("Descriptive statistics of the DataFrame after Standard scaling:")
display(df1.describe())

DataFrame after Standard scaling:


,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,...,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia,income_>50K
0,-0.999145,0.419934,-1.270758,0.0,0.0,-0.192863,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
1,-0.045382,-1.017089,-0.456714,0.0,0.0,1.424021,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
2,-0.779046,1.575412,0.764352,0.0,0.0,-0.192863,True,False,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0.394816,-0.277440,-0.049692,0.0,0.0,-0.192863,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
4,-1.512710,-0.873553,-0.049692,0.0,0.0,-1.405526,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False


Descriptive statistics of the DataFrame after Standard scaling:


,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week
count,4.884200e+04,4.884200e+04,4.884200e+04,48842.0,48842.0,4.884200e+04
mean,-6.721075e-17,9.608810e-17,-1.325303e-16,0.0,0.0,3.578754e-17
std,1.000010e+00,1.000010e+00,1.000010e+00,0.0,0.0,1.000010e+00
min,-1.586076e+00,-1.830379e+00,-2.288313e+00,0.0,0.0,-1.405526e+00
25%,-7.790458e-01,-7.261295e-01,-4.567143e-01,0.0,0.0,-1.928628e-01
50%,-1.187483e-01,-9.049010e-02,-4.969232e-02,0.0,0.0,-1.928628e-01
75%,6.882819e-01,5.336468e-01,7.643517e-01,0.0,0.0,6.155792e-01
max,2.889274e+00,2.423311e+00,2.392440e+00,0.0,0.0,1.828242e+00


1. Missing Values Handling:


Diabetes Dataset (df):
No missing values were identified in the df DataFrame. An initial check using df.isnull().sum().sum() confirmed a total of 0 missing values across the entire dataset. Therefore, no specific handling steps like imputation or removal were required for this dataset.

Adult Income Dataset (df1):

Initial inspection revealed that missing values were represented by '?' characters in the workclass, occupation, and native-country columns. These '?' values were first replaced with np.nan to be properly recognized as missing values.
After this replacement, missing values were found in:
workclass: 2799 missing values
occupation: 2809 missing values
native-country: 857 missing values
These missing values were then handled by imputing them with the mode (most frequent value) of their respective columns. Specifically, 'Private' was used for workclass, 'Prof-specialty' for occupation, and 'United-States' for native-country. After imputation, there were no remaining missing values in df1.




2. Categorical Columns and Encoding:

Diabetes Dataset (df):

The categorical columns identified were Gender and CLASS.
These were encoded using One-Hot Encoding via pd.get_dummies(df, columns=['Gender', 'CLASS'], drop_first=True). The drop_first=True argument was used to avoid multicollinearity. This created new binary columns like Gender_M, CLASS_N, CLASS_P, CLASS_Y.
A redundant Gender_f column (since Gender_M already captured the binary information) and a duplicate CLASS_Y (due to a trailing space in the original column name) were subsequently dropped to ensure data integrity.


Adult Income Dataset (df1):

The categorical columns identified were workclass, education, marital-status, occupation, relationship, race, gender, native-country, and income.
These were also encoded using One-Hot Encoding via pd.get_dummies(df1, columns=categorical_cols_df1, drop_first=True). This process significantly expanded the DataFrame by creating numerous new binary columns corresponding to the unique categories within each original categorical column.



3. Difference Between Min-Max Scaling and Standardization (Z-score Normalization):

Min-Max Scaling (Normalization):

Goal: Scales features to a fixed range, usually between 0 and 1. The formula is: X_scaled = (X - X_min) / (X_max - X_min).
Effect: Preserves the relationships among data points but changes the absolute values and the scale. All values will fall within the specified range.
When to use:
When features have different ranges and you want to ensure all features contribute equally, such as in algorithms that use distance calculations (e.g., K-Nearest Neighbors, Support Vector Machines).
When you have data with clear upper and lower bounds (e.g., image pixel intensities).
When you want to reduce the effect of outliers without removing them entirely, as outliers will be compressed into the target range.
Standardization (Z-score Normalization):

Goal: Scales features to have a mean of 0 and a standard deviation of 1. The formula is: X_scaled = (X - X_mean) / X_std.
Effect: Centers the data around the mean and scales it according to the standard deviation. It does not bound values to a specific range, so outliers can still have a larger impact.
When to use:
When features have varying scales and the algorithm assumes a Gaussian distribution or performs better when data is centered around zero (e.g., Linear Regression, Logistic Regression, Neural Networks).
When the data contains outliers that you don't want to compress into a small range, as standardization handles them by scaling them relative to the mean and standard deviation of the entire dataset.
When the algorithm relies on the magnitudes of feature values being comparable.
Key Difference: Min-Max Scaling produces values within a specific, bounded range, making it good for algorithms sensitive to input scale. Standardization produces values with a mean of zero and unit variance, making it suitable for algorithms that assume a normal distribution or are sensitive to feature centering, and it handles outliers differently by not compressing them into a fixed range.